# 00 Diagnóstico del laboratorio

Ejecuta antes de cualquier modificación. Comprueba el entorno, no cambia archivos.

In [ ]:
import os,sys,shutil,subprocess

HADOOP_HOME = "/opt/hadoop-3.3.6"
HADOOP_CONF_DIR = f"{HADOOP_HOME}/etc/hadoop"

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["HADOOP_CONF_DIR"] = HADOOP_CONF_DIR
os.environ["YARN_CONF_DIR"] = HADOOP_CONF_DIR

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

resultado = subprocess.run(
    [f"{HADOOP_HOME}/bin/hdfs",
     "getconf", "-confKey", "fs.defaultFS"],
    capture_output=True,
    text=True
)

print("HDFS configurado:", resultado.stdout.strip())
print("Python:", sys.executable)
print("Error de configuración:", resultado.stderr.strip())

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("AFP_V2_ETL")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version:", spark.version)
print("MASTER:", sc.master)
print(
    "HDFS default:",
    sc._jsc.hadoopConfiguration().get("fs.defaultFS")
)
print(
    "CATALOGO:",
    spark.conf.get("spark.sql.catalogImplementation")
)

# Comprobar ejecución
resultado = spark.range(1000).repartition(2).count()
print("Filas procesadas:", resultado)